# PegInsertionSide-v1 — Interactive Notebook

Play with scripted primitives, record rollouts, and debug peg alignment.

This notebook attempts to create the ManiSkill2 `PegInsertionSide-v1` environment, run a simple scripted policy, and record a video inline. Adapt keys and gains to your local installation and observations.

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import io
import base64
import numpy as np
import imageio
import gymnasium as gym
import torch
import sapien
from mani_skill.utils.structs.pose import Pose
from IPython.display import HTML, display

def display_video(path, width=640):
    mp4 = open(path,'rb').read()
    data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()
    html = f'<video width="{width}" controls><source src="{data_url}" type="video/mp4"></video>'
    display(HTML(html))

def frames_to_mp4(frames, path, fps=30):
    # print(len(frames))
    # frames = frames.numpy()
    # Write frames (H x W x C uint8) to an mp4 file using imageio
    print(type(frames), len(frames), frames[0].shape if frames else 'No frames')
    imageio.mimwrite(path, frames, fps=fps, macro_block_size=None)


/home/cjimenez/miniconda3/envs/tfm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
global_frames = []
class InsertionCurriculumWrapper(gym.Wrapper):
    """
    Intercepts env.reset() to execute a scripted batched policy that grasps 
    the peg from the table and aligns it with the hole before handing 
    control to the RL agent.
    """
    def __init__(self, env, setup_steps=90):
        super().__init__(env)
        self.setup_steps = setup_steps

    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        global_frames.append(capture_frame_from_env(self.env).cpu().numpy())
        device = obs.device if isinstance(obs, torch.Tensor) else "cpu"
        n_envs = getattr(self.env.unwrapped, "num_envs", 1)
        action_dim = self.env.action_space.shape[-1]
        
        for step in range(self.setup_steps):
            # 1. Get current gripper position
            tcp_pos = self.env.unwrapped.agent.tcp.pose.p
            
            # 2. Get target grasp pose from the environment's own reward logic
            # The env explicitly uses an offset of [-0.06, 0, 0] to grab the peg tail
            tgt_gripper_pos = (self.env.unwrapped.peg.pose * sapien.Pose([-0.06, 0, 0])).p

            
            # 3. Get pre-insertion pose
            # The hole points along the local +X axis of box_hole_pose. 
            # We hover 15cm (-0.15) outside the hole.
            pre_insert_pos = (self.env.unwrapped.box_hole_pose * sapien.Pose([-0.15, 0, 0])).p
            
            print('tgt_gripper_pos:', tgt_gripper_pos, 'pre_insert_pos:', pre_insert_pos)
            # 4. Phase-based state machine
            if step < 20:
                # Phase A: Hover 10cm directly above the peg
                target_pos = tgt_gripper_pos + torch.tensor([0, 0, 0.1], device=device)
                gripper_act = 1.0 # Open
            elif step < 40:
                # Phase B: Drop down to the peg tail
                target_pos = tgt_gripper_pos
                gripper_act = 1.0 # Open
            elif step < 60:
                # Phase C: Close the gripper firmly
                target_pos = tgt_gripper_pos
                gripper_act = -1.0 # Close
            else:
                # Phase D: Fly to the hole
                target_pos = pre_insert_pos
                gripper_act = -1.0 # Keep closed
                
            # Proportional (P) controller for smooth movement
            delta_pos = (target_pos - tcp_pos) * 5.0
            delta_pos = torch.clamp(delta_pos, -1.0, 1.0)
            
            # Construct the native action
            scripted_action = torch.zeros((n_envs, action_dim), device=device)
            scripted_action[:, :3] = delta_pos
            scripted_action[:, -1] = gripper_act
            
            # Step the underlying environment silently
            obs, _, _, _, info = self.env.step(scripted_action)
            global_frames.append(capture_frame_from_env(self.env).cpu().numpy())

        # Control is handed to the RL agent! The robot is holding the peg right in front of the hole.
        return obs, info
class ManiskillTeleportWrapper(gym.Wrapper):
    """
    Bypasses the P-controller entirely! Instantly teleports the peg into the robot's 
    hand, and teleports the box directly in front of the robot. 
    """
    def __init__(self, env, setup_steps=15):
        super().__init__(env)
        self.setup_steps = setup_steps

    def reset(self, seed=None, options=None):
        # 1. Reset the environment normally (randomizes shapes and sizes)
        obs, info = self.env.reset(seed=seed, options=options)
        global_frames.append(capture_frame_from_env(self.env).cpu().numpy())
        
        env_unwrapped = self.env.unwrapped
        device = obs.device if isinstance(obs, torch.Tensor) else "cpu"
        n_envs = getattr(env_unwrapped, "num_envs", 1)
        action_dim = self.env.action_space.shape[-1]
        
        # 2. Teleport the Peg into the gripper
        tcp_pose = env_unwrapped.agent.tcp.pose
        
        # The env's reward logic assumes a grasp offset of [-0.06, 0, 0] relative to the peg.
        # By inverting this, we put the peg perfectly inside the TCP.
        offset_p = torch.zeros((n_envs, 3), device=device)
        offset_p[:, 0] = 0.05
        peg_offset = Pose.create_from_pq(p=offset_p)
        
        new_peg_pose = tcp_pose * peg_offset
        env_unwrapped.peg.set_pose(new_peg_pose)
        
        # 3. Teleport the Box directly in front of the gripper
        peg_lengths = env_unwrapped.peg_half_sizes[:, 0]
        
        # Place the hole exactly 2cm (0.02) in front of the peg tip
        hole_offset_p = torch.zeros((n_envs, 3), device=device)
        hole_offset_p[:, 0] = 0.06 + peg_lengths + 0.08
        
        hole_target_pose = tcp_pose * Pose.create_from_pq(p=hole_offset_p)
        
        # Apply the inverse hole offset to perfectly position the outer box
        new_box_pose = hole_target_pose * env_unwrapped.box_hole_offsets.inv()
        env_unwrapped.box.set_pose(new_box_pose)
        
        # 4. Settle the physics (Close the gripper tightly)
        scripted_action = torch.zeros((n_envs, action_dim), device=device)
        scripted_action[:, -1] = -1.0 # Force gripper closed
        
        for _ in range(self.setup_steps):
            obs, _, _, _, info = self.env.step(scripted_action)
            global_frames.append(capture_frame_from_env(self.env).cpu().numpy())
    

        # Hand control to the RL Agent!
        return obs, info

In [3]:
def capture_frame_from_env(env):
    "Try several common render calls to get an RGB frame (H,W,3)"
    try:
        # common gym-style render
        frame = env.render()
        if frame is None:
            frame = None
    except Exception as e:
        print(f"gym render failed: {e}")
        frame = None
    if frame is None:
        try:
            # ManiSkill / MuJoCo direct render fallback (may vary by build)
            frame = env.sim.render(width=640, height=480, camera_name='agentview')
        except Exception as e:
            print(f"ManiSkill render failed: {e}")
            frame = None
    return frame[0]

def make_video_env():
    """Single CPU env with render_mode='rgb_array' for video probes."""
    import mani_skill.envs  # noqa: F401
    from hires_vic.envs.maniskill_riemannian import ManiSkillRiemannianWrapper
    use_spd_manifold = True
    use_lie_group = False
    use_llm_prior = False
    use_fixed = True
    env = gym.make(
        'PegInsertionSide-v1',
        num_envs=1,
        obs_mode="state",
        sim_backend="cpu",
        render_mode="rgb_array",
        max_episode_steps=200,
    )
    
    # env = InsertionCurriculumWrapper(env, setup_steps=90)
    env = ManiskillTeleportWrapper(env, setup_steps=15)

    env = ManiSkillRiemannianWrapper(
        env,
        use_spd=use_spd_manifold,
        use_lie_group=use_lie_group,
        use_diag=False,
        use_fixed=use_fixed,
        is_eval=True,
        use_llm_prior=False,
        use_sim2real_obs=True,
        task_metrics_fn=None,
    )
    return env


In [4]:

try:
    env  = make_video_env()
except Exception as e:
    raise RuntimeError('Failed to create ManiSkill3 PegInsertionSide-v1 environment', e)

out_dir = 'outputs'
out_path = os.path.join(out_dir, 'peginsertion_primitive.mp4')
print('Env created:', type(env))
obs = env.reset()


action_dim = int(env.action_space.shape[-1])
settle_action = np.zeros(action_dim, dtype=np.float32)
settle_action[-1] = -1.0 # Robosuite OPEN (+1.0)

for _ in range(20):
    try:
        obs, rew, terminated, truncated, info = env.step(settle_action)

        frame = capture_frame_from_env(env)
        if frame is not None:
            global_frames.append(frame)
        if terminated or truncated:
            break

    except Exception as e:
        print('Error during env step or render:', e)
        break

if global_frames:
    frames_to_mp4(global_frames, out_path, fps=30)
    print('Saved video to', out_path)
    display_video(out_path)
else:
    print('No frames captured; check env.render()')

    


[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)


Registering custom environments...
ManiSkillRiemannianWrapper | SPD=True Diag=False VIC=False Fixed=True LieGroup=False Sim2Real=True | action 8D→17D | obs 29D
Env created: <class 'hires_vic.envs.maniskill_riemannian.ManiSkillRiemannianWrapper'>
Error during env step or render: 'numpy.ndarray' object has no attribute 'clone'
<class 'list'> 16 (512, 512, 3)
Saved video to outputs/peginsertion_primitive.mp4


Notes:
- If the scripted policy does not find useful observation keys, inspect `obs` (the observation dict) and adapt `find_key` to the correct keys.
- ManiSkill2 environments often require specialized setup; consult ManiSkill2 docs if env creation fails.
- You can replace `simple_scripted_policy` with your own primitive function that teleports objects or uses lower-level sim APIs.